# LASSI: LLM-based Automated Self-correcting pipeline for translating parallel Scientific codes

Translates a HeCBench application between **CUDA** and **OpenMP (GPU offload)** with an LLM, then compiles and runs the result, feeding every compile or runtime error back to the LLM until the code works.

With the default settings this reproduces the pipeline of the original LASSI paper. Optional extras (an LLM judge, a results CSV, repeated trials) are off unless you switch them on.

LLM backends: **Ollama**, **LM Studio**, any **OpenAI-compatible** endpoint, or **Argonne Argo Gateway** (Argonne network only).

Benchmark sources come from [HeCBench](https://github.com/zjin-lcf/HeCBench/tree/master/src). To compile and run you need `nvcc` (CUDA) and/or `nvc++` from the [NVIDIA HPC SDK](https://developer.nvidia.com/hpc-sdk) (OpenMP GPU offload).

## Setup

In [ ]:

# Reload the lassi package automatically when its source changes (handy while developing)
%load_ext autoreload
%autoreload 2

import lassi
from lassi import Config, run_experiment
from lassi import llm_backends

_____
***STEP 1/2: Choose the LLM and the experiment.***

In [ ]:
# --- LLM connection (only the backend you select below is used) -------------------------------
llm_backends.configure(
    # ollama_host="http://localhost:11434",            # default if omitted
    # lmstudio_base_url="http://localhost:1234/v1",    # default if omitted
    # openai_base_url="https://api.openai.com/v1",     # any OpenAI-compatible endpoint; the key is
    #                                                  # read from the OPENAI_API_KEY environment variable
    # argo_url="https://<gateway-host>/argoapi/api/v1/resource/streamchat/",   # Argonne network only
    # argo_username="<your Argonne domain username>",  # not a secret
)

# --- The experiment ---------------------------------------------------------------------------
# Applications (see source_apps.json): matrix-rotate, jacobi, layout, atomicCost, dense-embedding,
#   pathfinder, bsearch, entropy, colorwheel, randomAccess
# Directions: CUDA -> OMP  or  OMP -> CUDA
cfg = Config(
    app="jacobi",
    source_lang="CUDA",
    target_lang="OMP",

    # llm_source: "ollama" | "lmstudio" | "openai_compatible" | "argo"
    llm_source="ollama",
    model="wizardcoder:33b-v1.1-q8_0",   # e.g. "codestral:22b-v0.1-q8_0", "deepseek-coder-v2:16b-lite-instruct-fp16"
    num_ctx=16384,                       # context window in tokens (ollama only)

    # Compiler for the TARGET language. The defaults are the paper's NVIDIA A100 settings:
    #   CUDA: nvcc  -std=c++14 -Xcompiler -Wall -arch=sm_80 -O3
    #   OMP:  nvc++ -Wall -O3 -Minfo -mp=gpu -gpu=cc80
    # Change the -arch / -gpu flags (and the compiler path, if it is not on your PATH) for your GPU.
    # compiler="nvc++",
    # compiler_flags="-Wall -O3 -Minfo -mp=gpu -gpu=cc80",

    # --- Optional extras (off by default; the defaults reproduce the paper) ---
    # use_judge=True,            # ask an LLM whether the translation is functionally equivalent
    # model_judge="...",         # judge model on the same llm_source (default: same as `model`)
    # record_results=True,       # append runtimes, judge verdict, etc. to a results CSV, and keep every
    #                            # attempt, LLM call and the environment under translated_code/.../records/
    # total_trials=3,            # repeat the whole pipeline, e.g. for pass@k
    # timing_repeats=3,          # time reference and translation 3 times each and use the mean (the paper's method)
    # max_self_corrections=15,   # give up on a trial after this many self-corrections (default: never)
    # run_timeout_s=300,         # stop any program that runs longer than this

    # --- No GPU? Translate CUDA -> OpenMP and run it on the CPU (compiles with g++ -fopenmp) ---
    # cpu_only=True,
)

_____
***STEP 2/2: Run the pipeline.***

In [ ]:
run_experiment(cfg)